# CT Segmentation with IDC Data

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial demonstrates how to build a CT segmentation pipeline using IDC data with paired DICOM Segmentation (SEG) annotations.

## What You'll Learn

1. Find CT series with paired segmentations in IDC
2. Download image-segmentation pairs
3. Load DICOM SEG files with MONAI
4. Build a complete segmentation training pipeline

## Prerequisites

- Completed Tutorial 01 (Getting Started)
- Basic understanding of medical image segmentation

## Setup environment

In [ ]:
!pip install -q monai idc-index pydicom itkwasm-dicom

## Setup imports

In [ ]:
import os
import tempfile

import torch
from idc_index import IDCClient
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    ToTensord,
)
from monai.data import Dataset, CacheDataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
import monai

monai.config.print_config()

## 1. Find CT Series with Segmentations

IDC contains many collections with DICOM Segmentation objects. We'll use `seg_index` to find paired data.

In [ ]:
# Initialize IDC client
client = IDCClient()

# Fetch seg_index for segmentation metadata
client.fetch_index("seg_index")

# Explore available segmentation algorithms
algorithms = client.sql_query("""
    SELECT 
        AlgorithmName,
        AlgorithmType,
        COUNT(*) as seg_count
    FROM seg_index
    WHERE AlgorithmName IS NOT NULL
    GROUP BY AlgorithmName, AlgorithmType
    ORDER BY seg_count DESC
    LIMIT 10
""")

print("Top segmentation algorithms in IDC:")
print(algorithms.to_string(index=False))

In [ ]:
# Find CT series with TotalSegmentator segmentations
# TotalSegmentator provides 104 anatomical structure labels
paired_data = client.sql_query("""
    SELECT
        src.SeriesInstanceUID as image_series_uid,
        seg.SeriesInstanceUID as seg_series_uid,
        src.collection_id,
        src.PatientID,
        src.BodyPartExamined,
        seg.AlgorithmName,
        seg.total_segments,
        ROUND(src.series_size_MB, 2) as image_size_mb
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    WHERE src.Modality = 'CT'
        AND seg.AlgorithmName LIKE '%TotalSegmentator%'
    ORDER BY src.series_size_MB ASC
    LIMIT 5
""")

print(f"Found {len(paired_data)} CT series with TotalSegmentator segmentations:")
print(paired_data[['collection_id', 'PatientID', 'total_segments', 'image_size_mb']].to_string(index=False))

## 2. Download Image-Segmentation Pairs

In [ ]:
# Create download directory
data_dir = tempfile.mkdtemp(prefix="idc_seg_")
print(f"Download directory: {data_dir}")

# Use first 3 pairs for this demo
demo_data = paired_data.head(3)

# Collect all UIDs to download
all_uids = list(demo_data['image_series_uid']) + list(demo_data['seg_series_uid'])

print(f"\nDownloading {len(all_uids)} series ({len(demo_data)} image-seg pairs)...")

client.download_from_selection(
    seriesInstanceUID=all_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"
)

print("Download complete!")

## 3. Prepare Data Dictionaries

Create data dictionaries with paired image and label paths.

In [ ]:
# Build data dictionaries for MONAI
data_dicts = []
for _, row in demo_data.iterrows():
    data_dicts.append({
        "image": os.path.join(data_dir, row['image_series_uid']),
        "label": os.path.join(data_dir, row['seg_series_uid']),
        "patient_id": row['PatientID'],
    })

print(f"Created {len(data_dicts)} data dictionaries")
print(f"\nExample:")
print(f"  Image: {data_dicts[0]['image'][:60]}...")
print(f"  Label: {data_dicts[0]['label'][:60]}...")

## 4. Understanding DICOM-SEG Format

DICOM Segmentation (DICOM-SEG) is a special DICOM format for storing segmentation masks:

- **Enhanced multiframe**: All slices stored in a single file (unlike regular DICOM series)
- **Segment metadata**: Each segment has label, description, and algorithm info
- **Cannot use ITKReader**: Standard DICOM readers don't support this format

**Important**: While `LoadImaged` with the default ITKReader works for CT images (regular DICOM series), DICOM-SEG files require a specialized reader. We use `LoadDicomSegd` from `idc_monai` which uses `itkwasm-dicom` (wrapping dcmqi) for robust DICOM-SEG reading with proper spatial metadata.

In [ ]:
from idc_monai.transforms import LoadDicomSegd

# Define transforms for training
train_transforms = Compose([
    # Load CT DICOM series (standard DICOM - ITKReader handles this)
    LoadImaged(keys=["image"]),
    
    # Load DICOM-SEG (requires specialized ITKWasm reader)
    LoadDicomSegd(keys=["label"]),
    
    EnsureChannelFirstd(keys=["image", "label"]),
    
    # Spatial preprocessing
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),  # bilinear for image, nearest for label
    ),
    
    # Intensity preprocessing for CT
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-175,
        a_max=250,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    
    # Crop to foreground
    CropForegroundd(keys=["image", "label"], source_key="image"),
    
    # Random crop for training (patch-based training)
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1,
        neg=1,
        num_samples=2,  # Extract 2 patches per volume
    ),
    
    # Data augmentation
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
])

# Validation transforms (no augmentation, no random crop)
val_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadDicomSegd(keys=["label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-175,
        a_max=250,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
])

## 5. Create Datasets and DataLoaders

In [ ]:
# Split data into train/val (use 2 for train, 1 for val in this demo)
train_files = data_dicts[:2]
val_files = data_dicts[2:]

print(f"Training samples: {len(train_files)}")
print(f"Validation samples: {len(val_files)}")

# Create datasets
# Using regular Dataset for this demo; use CacheDataset for faster training
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)

In [ ]:
# Load and inspect a sample
sample = train_ds[0]

print(f"Image shape: {sample['image'].shape}")
print(f"Label shape: {sample['label'].shape}")
print(f"Unique label values: {torch.unique(sample['label']).numpy()}")

## 6. Visualize Image-Label Pairs

DICOM SEG files contain color information for each segment in `recommendedDisplayRGBValue`. We can extract this from the metadata stored by `LoadDicomSegd` to render segments with their original colors.

In [ ]:
def build_seg_colormap(overlay_info):
    """Build a matplotlib colormap from DICOM SEG segment colors.
    
    Args:
        overlay_info: Dictionary from LoadDicomSegd containing segmentAttributes
        
    Returns:
        ListedColormap with colors for each segment label
    """
    segment_attrs = overlay_info.get("segmentAttributes", [[]])
    
    # Flatten segment attributes (may be nested in groups)
    all_segments = []
    for group in segment_attrs:
        all_segments.extend(group)
    
    if not all_segments:
        return plt.cm.nipy_spectral
    
    max_label = max(seg.get("labelID", 0) for seg in all_segments)
    
    # Build RGBA color array: index 0 = background (transparent)
    colors = np.zeros((max_label + 1, 4))
    colors[0] = [0, 0, 0, 0]  # Background transparent
    
    for seg in all_segments:
        label_id = seg.get("labelID", 0)
        rgb = seg.get("recommendedDisplayRGBValue", [128, 128, 128])
        colors[label_id] = [rgb[0]/255, rgb[1]/255, rgb[2]/255, 1.0]
    
    return ListedColormap(colors)

# Load validation sample for visualization (no random cropping)
val_sample = Dataset(data=data_dicts[:1], transform=val_transforms)[0]

# Visualize a slice with overlay using DICOM SEG colors
image = val_sample['image'][0].numpy()
label = val_sample['label'][0].numpy()

# Build colormap from DICOM SEG metadata
overlay_info = val_sample.get('label_meta_dict', {}).get('overlay_info', {})
seg_cmap = build_seg_colormap(overlay_info)

# Find slice with segmentation
z_slices_with_label = np.where(label.sum(axis=(0, 1)) > 0)[0]
if len(z_slices_with_label) > 0:
    z_mid = z_slices_with_label[len(z_slices_with_label) // 2]
else:
    z_mid = image.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Image only
axes[0].imshow(image[:, :, z_mid].T, cmap='gray', origin='lower')
axes[0].set_title('CT Image')
axes[0].axis('off')

# Label only (using DICOM SEG colors)
axes[1].imshow(label[:, :, z_mid].T, cmap=seg_cmap, origin='lower',
               vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
axes[1].set_title('Segmentation Labels\n(DICOM SEG colors)')
axes[1].axis('off')

# Overlay (using DICOM SEG colors)
axes[2].imshow(image[:, :, z_mid].T, cmap='gray', origin='lower')
mask = np.ma.masked_where(label[:, :, z_mid] == 0, label[:, :, z_mid])
axes[2].imshow(mask.T, cmap=seg_cmap, alpha=0.6, origin='lower',
               vmin=0, vmax=len(seg_cmap.colors)-1, interpolation='nearest')
axes[2].set_title('Overlay\n(DICOM SEG colors)')
axes[2].axis('off')

plt.suptitle(f'Axial Slice {z_mid}', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Define Model, Loss, and Optimizer

Using MONAI's UNet for multi-class segmentation.

In [ ]:
# Get number of classes from the labels
num_classes = int(torch.unique(sample['label']).max().item()) + 1
print(f"Number of classes: {num_classes}")

# For demo, we'll simplify to binary segmentation (background vs foreground)
# In practice, you'd use all classes or select specific ones
num_classes = 2  # Binary for this demo

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Create UNet model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=num_classes,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# Loss function
loss_function = DiceLoss(to_onehot_y=True, softmax=True)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Metric
dice_metric = DiceMetric(include_background=False, reduction="mean")

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Training Loop (Demo)

This is a simplified training loop for demonstration. In practice, you'd train for many more epochs.

In [ ]:
# Training parameters
max_epochs = 2  # Just 2 epochs for demo
val_interval = 1

# For binary segmentation, convert multi-class to binary
def to_binary(label):
    return (label > 0).float()

# Training loop
best_metric = -1
epoch_loss_values = []

for epoch in range(max_epochs):
    print(f"\nEpoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in train_loader:
        step += 1
        inputs = batch_data["image"].to(device)
        labels = to_binary(batch_data["label"]).to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"  Step {step}, Loss: {loss.item():.4f}")
    
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"  Epoch Loss: {epoch_loss:.4f}")

print("\nTraining complete!")

## 9. Cleanup

In [ ]:
# Optionally cleanup
# import shutil
# shutil.rmtree(data_dir)

print(f"Data saved at: {data_dir}")

## Summary

In this tutorial, you learned how to:

1. **Query IDC** for CT series with paired segmentations using `seg_index`
2. **Download** image-segmentation pairs from IDC
3. **Load DICOM-SEG** files using `LoadDicomSegd` (with ITKWasm backend)
4. **Build transforms** for segmentation training (image + label)
5. **Create a training pipeline** with UNet and Dice loss

## Key Points

- IDC has many collections with pre-computed segmentations (TotalSegmentator, nnU-Net, etc.)
- Use `seg_index` to find paired image-segmentation data
- Join `seg_index.segmented_SeriesInstanceUID` with `index.SeriesInstanceUID` to link segmentations to source images
- **DICOM-SEG requires special handling**: Use `LoadDicomSegd` instead of `LoadImaged` for segmentation files
- Standard ITKReader/SimpleITK cannot read DICOM-SEG format

## Next Steps

- **Tutorial 3**: Working with analysis results and different annotation types
- Explore TotalSegmentator's 104 anatomical labels
- Use `CacheDataset` for faster training on larger datasets

## Resources

- [TotalSegmentator Paper](https://arxiv.org/abs/2208.05868)
- [MONAI Segmentation Tutorials](https://github.com/Project-MONAI/tutorials/tree/main/3d_segmentation)
- [DICOM SEG Standard](https://dicom.nema.org/medical/dicom/current/output/chtml/part03/sect_C.8.20.html)
- [ITKWasm DICOM Documentation](https://wasm.itk.org/en/latest/introduction/file_formats/dicom.html)